# 卷积神经网络实验运行记录

这个 notebook 是我用来整理本次 CNN 作业运行过程的。主要代码还是放在 `src/` 目录里，这里不重新堆一遍代码，主要记录一下我是怎么跑实验、看结果和整理图表的。

## 1. 先检查一下环境

正式训练前，我先确认一下 PyTorch 和 GPU 是否能正常用。因为后面模型训练时间比较长，如果这里没有识别到 CUDA，就要先检查环境。

In [ ]:
import os
import sys
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("当前没有检测到 GPU，训练会比较慢。")

## 2. 看一下工程文件是否齐全

我的主要实现都放在 `src` 里面。这里先简单列一下文件，确认训练脚本、模型代码和数据处理代码都在。

In [ ]:
project_root = Path.cwd()
src_dir = project_root / "src"

print("当前目录:", project_root)
print("src 是否存在:", src_dir.exists())

if src_dir.exists():
    for file in sorted(src_dir.glob("*.py")):
        print("-", file.name)
else:
    print("没有找到 src 目录。需要把这个 notebook 放到项目根目录，也就是和 src 同一级。")

## 3. 训练命令

这部分是我实际跑实验时用到的命令。由于五个模型完整训练需要比较久，所以我没有让 notebook 一打开就自动训练，而是把命令单独放在这里，方便需要复现实验时直接运行。

如果在 Windows 上 DataLoader 卡住，可以把 `num_workers` 改成 0。

In [ ]:
# 训练全部模型
# 这条命令会依次训练 OriginalCNN、MicroResNet、MicroDenseNet、MicroMobileNet 和 MicroRes2Net

# !python src/run_all.py --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp

如果只想单独重跑某一个模型，可以用下面这种方式。比如只重跑 ResNet，就取消对应那一行前面的注释。

In [ ]:
# !python src/train.py --model original --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp
# !python src/train.py --model resnet --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp
# !python src/train.py --model densenet --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp
# !python src/train.py --model mobilenet --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp
# !python src/train.py --model res2net --epochs 120 --batch_size 128 --lr 0.05 --num_workers 0 --amp

## 4. 生成报告里用到的结果图

训练完成后，我用 `report_assets.py` 统一整理曲线图、模型结构和汇总表。这样报告里用到的图片和数据都从同一份输出目录来，比较不容易写错。

In [ ]:
# 训练完成后再运行这一行
# !python src/report_assets.py --output_dir outputs

## 5. 读取实验汇总结果

下面这段主要是看 `outputs/summary.csv`。如果已经训练完成，这里会显示每个模型的最佳验证准确率、测试准确率和参数量。

In [ ]:
import pandas as pd

summary_path = Path("outputs") / "summary.csv"

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
else:
    print("暂时没有找到 outputs/summary.csv。")
    print("如果还没训练，先运行上面的训练命令；如果已经训练过，检查 outputs 目录是否放在项目根目录下。")

## 6. 简单画一下模型对比图

我主要看测试准确率和参数量两个指标。准确率可以反映分类效果，参数量可以大致反映模型规模。

In [ ]:
import matplotlib.pyplot as plt

if summary_path.exists():
    summary = pd.read_csv(summary_path)

    # 兼容不同版本 summary.csv 的列名
    name_col = "model" if "model" in summary.columns else summary.columns[0]
    acc_candidates = ["test_acc", "Test Acc", "test_accuracy", "Test Accuracy"]
    param_candidates = ["params", "Parameters", "num_params"]

    acc_col = next((c for c in acc_candidates if c in summary.columns), None)
    param_col = next((c for c in param_candidates if c in summary.columns), None)

    if acc_col is not None:
        plt.figure(figsize=(8, 4))
        plt.bar(summary[name_col], summary[acc_col])
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Test Accuracy")
        plt.title("Model Test Accuracy")
        plt.tight_layout()
        plt.show()
    else:
        print("summary.csv 里没有找到测试准确率列，可以手动检查列名：", list(summary.columns))

    if param_col is not None:
        plt.figure(figsize=(8, 4))
        plt.bar(summary[name_col], summary[param_col])
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Parameters")
        plt.title("Model Size Comparison")
        plt.tight_layout()
        plt.show()
else:
    print("没有 summary.csv，暂时不能画模型对比图。")

## 7. 查看训练曲线

每个模型训练后都会保存 loss 和 accuracy 曲线。这里我把能找到的曲线图列出来，写报告时就从这些图里挑对应模型的结果。

In [ ]:
from IPython.display import Image, display

outputs_dir = Path("outputs")

if outputs_dir.exists():
    curve_files = sorted(outputs_dir.glob("*/*_curves.png"))

    if len(curve_files) == 0:
        print("outputs 目录里还没有找到曲线图。")
    else:
        for img_path in curve_files:
            print(img_path)
            display(Image(filename=str(img_path)))
else:
    print("没有找到 outputs 目录。")

## 8. 我自己的结果记录

这一部分用来对照报告里的结论。最终实验里，OriginalCNN 作为 baseline，准确率最低；ResNet 和 DenseNet 效果最好；MobileNet 参数和计算更轻；Res2Net 体现了多尺度特征的作用。

我在报告里主要根据这几点展开分析，没有只看最终准确率，而是把结构差异、训练曲线和模型规模一起比较。

In [ ]:
# 可以在这里手动记录最终结果，方便和报告核对
results_note = {
    "OriginalCNN": "Test Acc 约 73.99%，作为 baseline",
    "MicroResNet": "Test Acc 约 95.43%，整体效果最好",
    "MicroDenseNet": "Test Acc 约 95.31%，参数效率比较高",
    "MicroMobileNet": "Test Acc 约 93.26%，结构更轻量",
    "MicroRes2Net": "Test Acc 约 94.05%，加入了多尺度残差分支",
}

for model, note in results_note.items():
    print(f"{model}: {note}")

## 9. 提交前检查

最后提交前，我主要检查这几项：报告、源码、运行说明、结果曲线和汇总表是否都在。这样助教即使不重新训练，也能看到完整的实验过程和结果。

In [ ]:
need_check = [
    "实验报告.pdf",
    "requirements.txt",
    "README.md",
    "src",
    "outputs",
    "run_experiment.ipynb",
]

for item in need_check:
    path = Path(item)
    print(f"{item}: {'存在' if path.exists() else '未找到'}")